## Working with sheets

Sheets are tables of structured data held alongside a project. This notebook follows one sheet from start to finish: it creates a table of sample metadata, adds the samples, reads and summarizes them, corrects a few rows, and then deletes the sheet.

In [ ]:
from cirro import DataPortal

portal = DataPortal()
project = portal.get_project_by_name("Test Project")

#### Creating a sheet

Each column has a name and a data type. `sample_id` is `required`, so it cannot be null, and `unique`, so no two rows can share it.

`sheet_type` says whether the sheet is a `TABLE`, which holds rows you write to it, or a `VIEW`, which is materialized from a query. `table_name` is how SQL queries refer to the sheet, and `namespace_name` groups related tables and cannot be changed later.

In [ ]:
from cirro_api_client.v1.models import ColumnDataType, ColumnDef, SheetCreationMode, TableSheetInput

sheet = project.create_sheet(
    TableSheetInput(
        sheet_type="TABLE",
        name="Example Samples",
        description="Sample metadata used by the Sheets example",
        namespace_name="default",
        table_name="example_samples",
        sheet_creation_mode=SheetCreationMode.STANDARD,
        columns=[
            ColumnDef(name="sample_id", data_type=ColumnDataType.STRING, required=True, unique=True),
            ColumnDef(name="tissue", data_type=ColumnDataType.STRING),
            ColumnDef(name="condition", data_type=ColumnDataType.STRING),
            ColumnDef(name="read_count", data_type=ColumnDataType.BIGINT)
        ]
    )
)
print(sheet)

#### Inserting rows

Each row maps column names to values, and columns left out are null. S3's read count isn't in yet, so it is left out. To insert from a DataFrame, pass `df.to_dict(orient="records")`.

In [ ]:
inserted = sheet.insert_rows([
    {"sample_id": "S1", "tissue": "liver", "condition": "control", "read_count": 21000000},
    {"sample_id": "S2", "tissue": "liver", "condition": "treated", "read_count": 19500000},
    {"sample_id": "S3", "tissue": "lung", "condition": "control"},
    {"sample_id": "S4", "tissue": "kidney", "condition": "treated", "read_count": 18000000},
    {"sample_id": "S5", "tissue": "lung", "condition": "treated", "read_count": 22500000}
])
print(f"Inserted {inserted} rows")

#### Reading the sheet

`to_dataframe` reads the whole sheet. Its first column, `_row_id`, identifies each row, and is how the updates and deletes below pick out the rows to change.

In [ ]:
df = sheet.to_dataframe()
row_ids = {sample_id: int(row_id) for sample_id, row_id in zip(df["sample_id"], df["_row_id"])}
df

#### Querying with SQL

`query_sheets` runs SQL across the project's sheets and returns a DataFrame. Refer to a sheet by its `table_name`.

In [ ]:
project.query_sheets("""
    SELECT condition, COUNT(*) AS samples, AVG(read_count) AS mean_reads
    FROM example_samples
    GROUP BY condition
""")

#### Updating a row

S3's read count is in now. Only the columns you name are written; the rest of the row is left alone.

In [ ]:
sheet.update_rows({row_ids["S3"]: {"read_count": 20100000}})

#### Deleting a row

S5 failed QC, so remove it.

In [ ]:
sheet.delete_rows([row_ids["S5"]])

sheet.to_dataframe()

#### Deleting the sheet

Delete the sheet, along with its rows, once you are done with it.

In [ ]:
sheet.delete()